In [ ]:
# =============================================================================
# STEP 0: FETCH 2026 CHICAGO DATA  &  RETRAIN MODEL
#
# Pulls delta crime data from the City of Chicago SODA API, engineers features
# EXACTLY matching the original training pipeline (H3 res-8, EWMA momentum,
# city-normalised rolling stats, spatial neighbour lag, shift-date anchoring),
# retrains the XGBoost pipeline, and overwrites the deployment artefacts
# so that Steps 1-3 automatically use the refreshed model.
#
# ★ Run this cell each time you want to pull the latest data and retrain. ★
# =============================================================================

import os, json, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# ── Install / import dependencies ────────────────────────────────────────────
import subprocess, sys
for pkg in ["requests", "h3", "xgboost", "scikit-learn"]:
    try:
        __import__(pkg.replace("-", "_").split("[")[0])
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

import requests, h3
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)

DEPLOY_DIR  = "deployment"
API_URL     = "https://data.cityofchicago.org/resource/f6bk-yv3r.json"
CACHE_FILE  = os.path.join(DEPLOY_DIR, "_last_fetch_offset.json")
H3_RES      = 8                              # ★ Resolution 8 — matches training notebook

# ★ Violent crime types — exact match to training notebook
VIOLENT_TYPES = ["BATTERY", "ASSAULT", "ROBBERY"]

EPSILON = 1e-6

# ═════════════════════════════════════════════════════════════════════════════
# 1. FETCH DATA FROM SODA API (with pagination)
# ═════════════════════════════════════════════════════════════════════════════
def fetch_2026_crimes(api_url=API_URL, year=2026, batch_size=50000):
    """
    Pull all 2026 crime records via the SODA API.
    Uses $offset pagination to get everything, and caches the last-seen
    record count so the next run reports delta.
    """
    last_count = 0
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE) as f:
            state = json.load(f)
            last_count = state.get("total_rows", 0)

    all_rows = []
    offset = 0
    print(f"Fetching {year} crime data from Chicago SODA API …")

    while True:
        params = {
            "$where" : f"year='{year}'",
            "$limit" : batch_size,
            "$offset": offset,
            "$order" : "date ASC",
        }
        resp = requests.get(api_url, params=params, timeout=120)
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        all_rows.extend(batch)
        offset += len(batch)
        print(f"  … fetched {offset:,} rows so far")
        if len(batch) < batch_size:
            break

    df = pd.DataFrame(all_rows)
    delta = len(df) - last_count
    print(f"✓ Total {year} records: {len(df):,}  (Δ {delta:+,} new since last run)")

    os.makedirs(DEPLOY_DIR, exist_ok=True)
    with open(CACHE_FILE, "w") as f:
        json.dump({"total_rows": len(df), "fetched_at": str(datetime.now())}, f)

    return df


# ═════════════════════════════════════════════════════════════════════════════
# 2. FEATURE ENGINEERING — exact replica of training notebook
# ═════════════════════════════════════════════════════════════════════════════

def engineer_features(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Transform raw API rows into the labelled, feature-rich DataFrame
    that EXACTLY matches the original training pipeline.
    """
    df = raw.copy()

    # ── 2a. Basic cleaning ────────────────────────────────────────────────
    df["Date"]      = pd.to_datetime(df["date"], errors="coerce")
    df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df["primary_type"] = df["primary_type"].astype(str).str.upper().str.strip()
    df = df.dropna(subset=["Date", "latitude", "longitude"])

    # ── 2b. Filter to violent crimes only ─────────────────────────────────
    rows_before = len(df)
    df = df[df["primary_type"].isin(VIOLENT_TYPES)].copy()
    print(f"  Filtered to violent crimes: {rows_before:,} → {len(df):,}")

    # ── 2c. Chicago boundary filter ───────────────────────────────────────
    chicago_mask = (
        (df["latitude"] > 41.6) & (df["latitude"] < 42.1) &
        (df["longitude"] > -88.0) & (df["longitude"] < -87.5)
    )
    df = df[chicago_mask].copy()

    # ── 2d. H3 tile assignment (resolution 8) ─────────────────────────────
    df["h3_address"] = df.apply(
        lambda r: h3.latlng_to_cell(r["latitude"], r["longitude"], H3_RES), axis=1
    )

    # ── 2e. Shift assignment — exact match to training notebook ───────────
    # np.select with hour.between(6,13) / hour.between(14,21) / default overnight
    hour = df["Date"].dt.hour
    df["shift"] = np.select(
        [hour.between(6, 13), hour.between(14, 21)],
        ["morning_noon", "afternoon_night"],
        default="overnight"
    )
    # shift_date: anchor overnight pre-6am crimes to previous calendar day
    df["shift_date"] = df["Date"].dt.floor("D")
    df.loc[hour < 6, "shift_date"] -= pd.Timedelta(days=1)

    # ── 2f. Build master grid: tile × date × shift (zero-fill) ────────────
    tile_shift_counts = (
        df.groupby(["h3_address", "shift_date", "shift"])
        .size()
        .reset_index(name="crime_count")
    )
    shift_order = ["morning_noon", "afternoon_night", "overnight"]
    idx = pd.MultiIndex.from_product(
        [df["h3_address"].unique(),
         pd.date_range(df["shift_date"].min(), df["shift_date"].max(), freq="D"),
         shift_order],
        names=["h3_address", "shift_date", "shift"]
    )
    master_grid = pd.DataFrame(index=idx).reset_index()
    final_df = pd.merge(
        master_grid, tile_shift_counts,
        on=["h3_address", "shift_date", "shift"], how="left"
    )
    final_df["crime_count"] = final_df["crime_count"].fillna(0)
    final_df["target"] = (final_df["crime_count"] > 0).astype(int)

    # Add Date column (shift_date + shift start hour)
    shift_start_hour = {"morning_noon": 6, "afternoon_night": 14, "overnight": 22}
    final_df["Date"] = (
        final_df["shift_date"]
        + pd.to_timedelta(final_df["shift"].map(shift_start_hour), unit="h")
    )

    n_tiles = final_df["h3_address"].nunique()
    print(f"  Master grid: {len(final_df):,} rows × {n_tiles:,} tiles")

    # ── 2g. lag_1d: same-shift 1-day lag ──────────────────────────────────
    final_df = final_df.sort_values(["h3_address", "shift", "shift_date"]).reset_index(drop=True)
    final_df["lag_1d"] = (
        final_df.groupby(["h3_address", "shift"])["crime_count"].shift(1)
    )

    # ── 2h. Rolling averages (grouped by tile AND shift) ──────────────────
    final_df["rolling_7d_mean"] = (
        final_df.groupby(["h3_address", "shift"])["crime_count"]
        .transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())
    )
    final_df["rolling_30d_mean"] = (
        final_df.groupby(["h3_address", "shift"])["crime_count"]
        .transform(lambda x: x.shift(1).rolling(window=30, min_periods=1).mean())
    )

    # ── 2i. Tile Crime Density Percentile (EWMA-based) ────────────────────
    daily_tile = (
        final_df.groupby(["h3_address", "shift_date"])["crime_count"]
        .sum().reset_index()
        .sort_values(["h3_address", "shift_date"])
        .reset_index(drop=True)
    )
    daily_tile["tile_ewma_crime"] = (
        daily_tile.groupby("h3_address")["crime_count"]
        .transform(lambda x: x.shift(1).ewm(halflife=30, min_periods=1).mean())
    )
    daily_tile["tile_crime_density_percentile"] = (
        daily_tile.groupby("shift_date")["tile_ewma_crime"]
        .transform(lambda x: x.rank(pct=True))
        .fillna(0.5)
    )

    # ── 2j. Tile Momentum (fast EWMA / slow EWMA) ────────────────────────
    daily_tile["tile_ewma_fast"] = (
        daily_tile.groupby("h3_address")["crime_count"]
        .transform(lambda x: x.shift(1).ewm(halflife=7, min_periods=1).mean())
    )
    daily_tile["tile_ewma_slow"] = (
        daily_tile.groupby("h3_address")["crime_count"]
        .transform(lambda x: x.shift(1).ewm(halflife=30, min_periods=1).mean())
    )
    daily_tile["tile_momentum"] = (
        daily_tile["tile_ewma_fast"] / (daily_tile["tile_ewma_slow"] + EPSILON)
    ).fillna(1.0)

    # Merge daily features onto final_df
    final_df = final_df.merge(
        daily_tile[["h3_address", "shift_date",
                    "tile_crime_density_percentile", "tile_momentum"]],
        on=["h3_address", "shift_date"], how="left"
    )

    # ── 2k. Cyclical time features — exact formulas from training ─────────
    final_df["day_of_week"] = final_df["Date"].dt.dayofweek
    final_df["month"]       = final_df["Date"].dt.month

    final_df["day_sin"]   = np.sin(2 * np.pi * final_df["day_of_week"] / 7)
    final_df["day_cos"]   = np.cos(2 * np.pi * final_df["day_of_week"] / 7)
    # ★ (month - 1) / 12  — matches training notebook exactly
    final_df["month_sin"] = np.sin(2 * np.pi * (final_df["month"] - 1) / 12)
    final_df["month_cos"] = np.cos(2 * np.pi * (final_df["month"] - 1) / 12)

    # Shift dummies
    final_df["is_afternoon_night"] = (final_df["shift"] == "afternoon_night").astype(int)
    final_df["is_overnight"]       = (final_df["shift"] == "overnight").astype(int)

    # ── 2l. Spatial lag: neighbor_lag_1d ──────────────────────────────────
    print("  Computing spatial neighbour lag (this may take a minute) …")
    final_df_sorted = final_df.sort_values(["h3_address", "Date"]).reset_index(drop=True)
    prev_crime = (
        final_df_sorted.groupby("h3_address")["crime_count"]
        .shift(1).fillna(0)
    )
    prev_lookup = dict(zip(
        zip(final_df_sorted["Date"], final_df_sorted["h3_address"]),
        prev_crime
    ))

    def calc_spatial_lag(row):
        neighbors = h3.grid_disk(row["h3_address"], k=1)
        neighbors = [n for n in neighbors if n != row["h3_address"]]
        return sum(prev_lookup.get((row["Date"], n), 0) for n in neighbors)

    final_df["neighbor_lag_1d"] = final_df.apply(calc_spatial_lag, axis=1)

    # ── 2m. City-normalised rolling features ──────────────────────────────
    city_daily_mean = (
        final_df.groupby("shift_date")["crime_count"]
        .mean().reset_index()
        .rename(columns={"crime_count": "city_daily_mean"})
        .sort_values("shift_date")
    )
    city_daily_mean["city_baseline"] = (
        city_daily_mean["city_daily_mean"]
        .shift(1).expanding().mean()
    )
    city_daily_mean["city_baseline"] = city_daily_mean["city_baseline"].fillna(
        city_daily_mean["city_daily_mean"].iloc[0]
    )
    final_df = final_df.merge(
        city_daily_mean[["shift_date", "city_baseline"]],
        on="shift_date", how="left"
    )

    final_df["rolling_30d_mean_norm"] = (
        final_df["rolling_30d_mean"] / (final_df["city_baseline"] + EPSILON)
    )
    final_df["rolling_7d_mean_norm"] = (
        final_df["rolling_7d_mean"] / (final_df["city_baseline"] + EPSILON)
    )
    final_df["neighbor_lag_1d_norm"] = (
        final_df["neighbor_lag_1d"] / (final_df["city_baseline"] + EPSILON)
    )

    # Fill remaining NaN from first-day cold-start
    for col in ["lag_1d", "rolling_7d_mean", "rolling_30d_mean",
                "rolling_7d_mean_norm", "rolling_30d_mean_norm",
                "neighbor_lag_1d_norm", "tile_crime_density_percentile",
                "tile_momentum"]:
        final_df[col] = final_df[col].fillna(0)

    print(f"  ✓ Feature engineering complete — {len(final_df):,} rows")
    return final_df, daily_tile


# ═════════════════════════════════════════════════════════════════════════════
# 3. RETRAIN THE XGBOOST MODEL — same pipeline as training notebook
# ═════════════════════════════════════════════════════════════════════════════

# ★ Exact feature columns from training notebook
CATEGORICAL_COLS = ["is_afternoon_night", "is_overnight"]
NUMERIC_COLS = [
    "lag_1d",
    "rolling_7d_mean_norm",
    "rolling_30d_mean_norm",
    "tile_crime_density_percentile",
    "tile_momentum",
    "day_sin", "day_cos",
    "month_sin", "month_cos",
    "neighbor_lag_1d_norm",
]
FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS


def retrain_model(final_df: pd.DataFrame, daily_tile: pd.DataFrame,
                  deploy_dir=DEPLOY_DIR):
    """
    Train a calibrated XGBoost with the same ColumnTransformer pipeline
    as the original training notebook.
    """
    # Drop rows with NaN in features or target
    df = final_df.dropna(subset=FEATURE_COLS + ["target"]).copy()

    X = df[FEATURE_COLS]
    y = df["target"]

    print(f"\nTraining data: {len(df):,} tile-shift-days "
          f"across {df['h3_address'].nunique():,} tiles")
    print(f"  Positive rate (violent crime): {y.mean():.3%}")

    # ── Time-based train/test split (80/20 by date) to avoid temporal leakage ───────────────────────
    split_date = df["shift_date"].quantile(0.8) # finds the date at the 80th percentile of the time range
    train_mask = df["shift_date"] <= split_date
    test_mask  = ~train_mask

    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]
    print(f"  Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")

    # ── ColumnTransformer — matches training notebook ─────────────────────
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OrdinalEncoder(
                handle_unknown="use_encoded_value", unknown_value=-1
            ), CATEGORICAL_COLS),
            ("num", "passthrough", NUMERIC_COLS),
        ],
        remainder="drop"
    )

    # ── XGBoost base model ────────────────────────────────────────────────
    base_xgb = XGBClassifier(
        n_estimators     = 300,
        max_depth        = 5,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1),
        eval_metric      = "logloss",
        random_state     = 42,
        use_label_encoder= False,
    )

    # ── Assemble pipeline: preprocessor → XGBoost ─────────────────────────
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", base_xgb),
    ])

    # ── Calibrate with CalibratedClassifierCV ─────────────────────────────
    calibrated = CalibratedClassifierCV(pipe, cv=3, method="sigmoid")
    calibrated.fit(X_train, y_train)

    # ── Evaluate ──────────────────────────────────────────────────────────
    y_prob = calibrated.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob) if y_test.nunique() > 1 else 0.0
    print(f"\n✓ ROC-AUC on hold-out: {auc:.4f}")

    # Find threshold targeting ~15-20% precision with reasonable recall
    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
    viable = np.where(precision[:-1] >= 0.12)[0]
    if len(viable) > 0:
        best_idx  = viable[np.argmax(recall[viable])]
        threshold = float(thresholds[best_idx])
    else:
        threshold = 0.15
    print(f"  Selected threshold: {threshold:.3f}")

    # ── Save deployment artefacts ─────────────────────────────────────────
    os.makedirs(deploy_dir, exist_ok=True)

    # 1) Calibrated pipeline
    joblib.dump(calibrated, os.path.join(deploy_dir, "xgb_calibrated_pipeline.joblib"))

    # 2) Tile baseline — exact columns from training notebook deployment cell
    last_date = final_df["shift_date"].max()
    tile_baseline = (
        final_df[final_df["shift_date"] == last_date]
        .groupby("h3_address")
        .agg(
            rolling_30d_mean_norm         = ("rolling_30d_mean_norm", "mean"),
            rolling_7d_mean_norm          = ("rolling_7d_mean_norm", "mean"),
            tile_crime_density_percentile = ("tile_crime_density_percentile", "mean"),
            tile_momentum                 = ("tile_momentum", "mean"),
            neighbor_lag_1d_norm          = ("neighbor_lag_1d_norm", "mean"),
            lag_1d                        = ("lag_1d", "mean"),
        )
        .reset_index()
    )
    tile_baseline.to_csv(os.path.join(deploy_dir, "tile_baseline.csv"), index=False)

    # 3) Metadata JSON
    base_rate = float(y.mean())
    meta = {
        "model"             : "XGBoost",
        "roc_auc"           : round(auc, 4),
        "threshold"         : round(threshold, 4),
        "base_rate"         : round(base_rate, 5),
        "feature_cols"      : FEATURE_COLS,
        "categorical_cols"  : CATEGORICAL_COLS,
        "numeric_cols"      : NUMERIC_COLS,
        "total_tiles"       : int(tile_baseline["h3_address"].nunique()),
        "n_train_rows"      : int(len(X_train)),
        "n_test_rows"       : int(len(X_test)),
        "trained_on"        : "2026 (API delta)",
        "trained_at"        : str(datetime.now()),
        "data_source"       : API_URL,
        "shift_hours"       : {
            "morning_noon"   : "06:00-13:59",
            "afternoon_night": "14:00-21:59",
            "overnight"      : "22:00-05:59",
        },
    }
    with open(os.path.join(deploy_dir, "metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"\n✓ Deployment artefacts saved to '{deploy_dir}/':")
    print(f"  • xgb_calibrated_pipeline.joblib")
    print(f"  • tile_baseline.csv  ({len(tile_baseline):,} tiles)")
    print(f"  • metadata.json")
    print(f"\n→ Now run STEP 1 to load the refreshed model.\n")

    return calibrated, meta


# ═════════════════════════════════════════════════════════════════════════════
# 4. RUN IT
# ═════════════════════════════════════════════════════════════════════════════
raw_df = fetch_2026_crimes()
final_df, daily_tile = engineer_features(raw_df)
model, metadata = retrain_model(final_df, daily_tile)


Fetching 2026 crime data from Chicago SODA API …
  … fetched 36,520 rows so far
✓ Total 2026 records: 36,520  (Δ +0 new since last run)
  Filtered to violent crimes: 36,469 → 10,680
  Master grid: 147,534 rows × 734 tiles
  Computing spatial neighbour lag (this may take a minute) …
  ✓ Feature engineering complete — 147,534 rows

Training data: 147,534 tile-shift-days across 734 tiles
  Positive rate (violent crime): 6.636%
  Train: 118,908 rows | Test: 28,626 rows


c:\Users\tpi_5\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:26:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\tpi_5\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:26:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\tpi_5\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:26:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



✓ ROC-AUC on hold-out: 0.7215
  Selected threshold: 0.071

✓ Deployment artefacts saved to 'deployment/':
  • xgb_calibrated_pipeline.joblib
  • tile_baseline.csv  (734 tiles)
  • metadata.json

→ Now run STEP 1 to load the refreshed model.



In [2]:
# =============================================================================
# STEP 1: INFERENCE ENGINE
# Load packaged model and predict crime probability for any tile-shift
#
# ★ Updated to match the retrained pipeline from STEP 0:
#   - Uses ColumnTransformer (categorical + numeric) feature order
#   - month_sin/cos use (month-1)/12 formula
#   - No is_late_night (removed from feature set)
#   - Tile baseline columns match training notebook deployment
# =============================================================================

import joblib
import json
import numpy as np
import pandas as pd
from datetime import datetime, date

class CrimePredictionEngine:
    """
    Loads the packaged XGBoost deployment artefacts and scores
    any tile x date x shift combination.

    Usage:
        engine = CrimePredictionEngine("deployment")
        results = engine.predict(query_date="2025-03-15", shift="afternoon_night")
    """

    SHIFT_MAP = {
        "morning_noon"   : {"is_afternoon_night": 0, "is_overnight": 0, "hour_start": 6},
        "afternoon_night": {"is_afternoon_night": 1, "is_overnight": 0, "hour_start": 14},
        "overnight"      : {"is_afternoon_night": 0, "is_overnight": 1, "hour_start": 22},
    }

    def __init__(self, deploy_dir="deployment"):
        print("Loading deployment package...")
        self.pipeline  = joblib.load(f"{deploy_dir}/xgb_calibrated_pipeline.joblib")
        self.baselines = pd.read_csv(f"{deploy_dir}/tile_baseline.csv")
        with open(f"{deploy_dir}/metadata.json") as f:
            self.meta = json.load(f)

        self.threshold    = self.meta["threshold"]
        self.feature_cols = self.meta["feature_cols"]
        print(f"✓ Model loaded  — ROC-AUC {self.meta['roc_auc']} | "
              f"Threshold {self.threshold}")
        print(f"✓ {len(self.baselines):,} tiles available for scoring")

    # ── Feature construction ──────────────────────────────────────────────────
    def _build_features(self, query_date: date, shift: str,
                        override_tiles: pd.DataFrame = None) -> pd.DataFrame:
        """
        Build the 12-feature inference row for each tile.
        Uses tile_baseline.csv for spatial/momentum features.
        Computes temporal features from query_date and shift.
        """
        if shift not in self.SHIFT_MAP:
            raise ValueError(f"shift must be one of: {list(self.SHIFT_MAP.keys())}")

        tiles = self.baselines.copy()

        # Apply any fresh lag overrides (e.g. from live data feed)
        if override_tiles is not None:
            tiles = tiles.merge(
                override_tiles[['h3_address', 'lag_1d']].rename(
                    columns={'lag_1d': 'lag_1d_fresh'}),
                on='h3_address', how='left'
            )
            tiles['lag_1d'] = tiles['lag_1d_fresh'].fillna(tiles['lag_1d'])
            tiles.drop(columns=['lag_1d_fresh'], inplace=True)

        # Shift flags
        s = self.SHIFT_MAP[shift]
        tiles['is_afternoon_night'] = s['is_afternoon_night']
        tiles['is_overnight']       = s['is_overnight']

        # Cyclical time features from query_date
        # ★ Uses (month - 1) / 12 to match training notebook
        dt  = pd.Timestamp(query_date)
        dow = dt.dayofweek   # 0=Monday … 6=Sunday
        mon = dt.month       # 1–12

        tiles['day_sin']   = np.sin(2 * np.pi * dow / 7)
        tiles['day_cos']   = np.cos(2 * np.pi * dow / 7)
        tiles['month_sin'] = np.sin(2 * np.pi * (mon - 1) / 12)
        tiles['month_cos'] = np.cos(2 * np.pi * (mon - 1) / 12)

        return tiles

    # ── Predict ───────────────────────────────────────────────────────────────
    def predict(
        self,
        query_date,
        shift,
        top_n          = 20,
        override_tiles = None,
        return_all     = False
    ) -> pd.DataFrame:
        """
        Score all tiles for a given date and shift.

        Parameters
        ----------
        query_date     : str or date, e.g. "2025-03-15"
        shift          : "morning_noon" | "afternoon_night" | "overnight"
        top_n          : number of highest-risk tiles to return (default 20)
        override_tiles : DataFrame with [h3_address, lag_1d] for fresh data
        return_all     : if True return all tiles, not just top_n flagged ones

        Returns
        -------
        DataFrame sorted by crime_probability descending
        """
        if isinstance(query_date, str):
            query_date = pd.Timestamp(query_date).date()

        tiles = self._build_features(query_date, shift, override_tiles)
        X     = tiles[self.feature_cols]

        probs  = self.pipeline.predict_proba(X)[:, 1]
        flagged = (probs >= self.threshold).astype(int)

        results = tiles[['h3_address']].copy()
        results['crime_probability'] = probs.round(4)
        results['flagged']           = flagged
        results['risk_tier'] = pd.cut(
            probs,
            bins  = [0, 0.10, 0.20, 0.35, 1.0],
            labels= ["Low", "Moderate", "High", "Critical"]
        )
        results['shift']      = shift
        results['query_date'] = str(query_date)
        results = results.sort_values('crime_probability', ascending=False)

        if return_all:
            return results.reset_index(drop=True)

        # Return flagged tiles OR top_n, whichever is larger
        flagged_df = results[results['flagged'] == 1]
        if len(flagged_df) < top_n:
            return results.head(top_n).reset_index(drop=True)
        return flagged_df.head(top_n).reset_index(drop=True)

    # ── Summary ───────────────────────────────────────────────────────────────
    def summarise(self, results: pd.DataFrame):
        """Print operational summary for a scored tile set."""
        n_flagged  = results['flagged'].sum()
        n_total    = len(self.baselines)
        n_critical = (results['risk_tier'] == 'Critical').sum()
        n_high     = (results['risk_tier'] == 'High').sum()

        print("\n" + "="*55)
        print(f"PATROL DISPATCH SUMMARY")
        print(f"Date  : {results['query_date'].iloc[0]}")
        print(f"Shift : {results['shift'].iloc[0]}")
        print("="*55)
        print(f"Tiles flagged for patrol : {n_flagged:>5,} / {n_total:,} tiles")
        print(f"  Critical risk (>35%)   : {n_critical:>5,}")
        print(f"  High risk    (20-35%)  : {n_high:>5,}")
        base_rate = self.meta.get('base_rate', 0.067)
        precision_est = self.meta.get('precision', 0.197)
        recall_est    = self.meta.get('recall', 0.437)
        print(f"Model precision at {self.threshold:.3f} : ~{precision_est:.1%}  "
              f"(1 in {int(1/precision_est)} dispatches finds a crime)")
        print(f"Expected crimes caught   : ~{int(n_flagged * precision_est * recall_est):,} "
              f"of ~{int(n_total * 3 * base_rate):,} "
              f"daily city-wide")
        print("="*55)


# ── Instantiate once ─────────────────────────────────────────────────────────
engine = CrimePredictionEngine("deployment")


Loading deployment package...
✓ Model loaded  — ROC-AUC 0.7215 | Threshold 0.0713
✓ 734 tiles available for scoring


In [ ]:
# =============================================================================
# STEP 2: INTERACTIVE USER INPUT — predict which tiles have crime
# Run this cell to enter your own date and shift and see predictions
# Note that this is a simple proof-of-concept interface for demonstration purposes.
# The predictions are based on a static model and baseline data, so they won't reflect real-time changes.
# E.g. if you select today's date, the lag_1d feature is still based on the historical baseline, not live data.
# =============================================================================

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── UI widgets ────────────────────────────────────────────────────────────────
date_picker = widgets.DatePicker(
    description = "Date:",
    value       = pd.Timestamp.today().date(),
    layout      = widgets.Layout(width="280px")
)

shift_dropdown = widgets.Dropdown(
    options     = [
        ("Morning / Noon  (06:00–13:59)", "morning_noon"),
        ("Afternoon / Night (14:00–21:59)", "afternoon_night"),
        ("Overnight  (22:00–05:59)", "overnight"),
    ],
    value       = "afternoon_night",
    description = "Shift:",
    layout      = widgets.Layout(width="380px")
)

top_n_slider = widgets.IntSlider(
    value=20, min=5, max=50, step=5,
    description="Top N tiles:",
    layout=widgets.Layout(width="380px")
)

run_button = widgets.Button(
    description  = "▶  Run Prediction",
    button_style = "primary",
    layout       = widgets.Layout(width="200px", height="36px")
)

out = widgets.Output()

# ── Risk tier colour map for display ─────────────────────────────────────────
TIER_COLORS = {
    "Critical": "\033[91m",   # red
    "High"    : "\033[93m",   # yellow
    "Moderate": "\033[94m",   # blue
    "Low"     : "\033[92m",   # green
}
RESET = "\033[0m"

# ── On-click handler ──────────────────────────────────────────────────────────
def on_run(b):
    with out:
        clear_output(wait=True)

        qdate = date_picker.value
        shift = shift_dropdown.value
        top_n = top_n_slider.value

        if qdate is None:
            print("Please select a date.")
            return

        print(f"Scoring {len(engine.baselines):,} tiles for "
              f"{qdate}  |  shift: {shift} ...")

        results = engine.predict(
            query_date = qdate,
            shift      = shift,
            top_n      = top_n,
            return_all = False
        )

        # ── Operational summary ───────────────────────────────────────────────
        engine.summarise(results)

        # ── Top N table ───────────────────────────────────────────────────────
        print(f"\nTop {top_n} Highest-Risk Tiles:")
        print(f"{'Rank':<5} {'H3 Tile':<20} {'Probability':>12} "
              f"{'Risk Tier':<12} {'Flag'}")
        print("-" * 58)

        for rank, (_, row) in enumerate(results.iterrows(), 1):
            tier   = str(row['risk_tier'])
            flag   = "🚨 DISPATCH" if row['flagged'] else "   monitor"
            color  = TIER_COLORS.get(tier, "")
            print(f"{rank:<5} {row['h3_address']:<20} "
                  f"{row['crime_probability']:>12.4f} "
                  f"{color}{tier:<12}{RESET} {flag}")

        # ── Bar chart of top tiles ────────────────────────────────────────────
        tier_palette = {
            "Critical": "#C0392B",
            "High"    : "#E67E22",
            "Moderate": "#2980B9",
            "Low"     : "#27AE60"
        }

        fig, axes = plt.subplots(1, 2, figsize=(16, max(6, top_n * 0.35)))
        fig.subplots_adjust(wspace=0.45, left=0.28, right=0.97,
                            top=0.90, bottom=0.08)

        # Panel 1: Crime probability per tile
        bar_colors = [tier_palette.get(str(t), "#95A5A6")
                      for t in results['risk_tier']]
        axes[0].barh(
            results['h3_address'][::-1],
            results['crime_probability'][::-1],
            color=list(reversed(bar_colors)),
            alpha=0.85, edgecolor='white', height=0.7
        )
        axes[0].axvline(
            engine.threshold, color='black',
            linestyle='--', linewidth=1.5,
            label=f"Dispatch threshold ({engine.threshold})"
        )
        axes[0].set_xlabel("Crime Probability (calibrated)", fontsize=10)
        axes[0].set_title(
            f"Top {top_n} Highest-Risk Tiles\n"
            f"{qdate}  |  {shift.replace('_',' ').title()}",
            fontweight='bold', fontsize=11
        )
        axes[0].legend(fontsize=8)
        axes[0].tick_params(axis='y', labelsize=7.5)
        axes[0].grid(axis='x', alpha=0.3)
        axes[0].set_xlim(0, max(results['crime_probability'].max() * 1.2, 0.3))

        # Panel 2: Risk tier distribution across ALL tiles
        all_results = engine.predict(
            query_date = qdate,
            shift      = shift,
            return_all = True
        )
        tier_counts = all_results['risk_tier'].value_counts().reindex(
            ["Critical", "High", "Moderate", "Low"], fill_value=0
        )
        tier_bar_colors = [tier_palette[t] for t in tier_counts.index]
        bars = axes[1].bar(
            tier_counts.index, tier_counts.values,
            color=tier_bar_colors, alpha=0.85,
            edgecolor='white', width=0.6
        )
        for bar, val in zip(bars, tier_counts.values):
            pct = val / len(all_results) * 100
            axes[1].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + len(all_results) * 0.005,
                f"{val:,}\n({pct:.1f}%)",
                ha='center', fontsize=9, fontweight='bold'
            )
        axes[1].set_ylim(0, tier_counts.max() * 1.25)
        axes[1].set_ylabel("Number of Tiles", fontsize=10)
        axes[1].set_title(
            f"Risk Tier Distribution — All {len(all_results):,} Tiles\n"
            f"{qdate}  |  {shift.replace('_',' ').title()}",
            fontweight='bold', fontsize=11
        )
        axes[1].grid(axis='y', alpha=0.3)

        legend_patches = [
            mpatches.Patch(color=tier_palette[t], label=t)
            for t in ["Critical", "High", "Moderate", "Low"]
        ]
        axes[1].legend(handles=legend_patches, fontsize=8, loc='upper right')

        plt.show()

        # ── CSV export for patrol briefing ────────────────────────────────────
        export_path = (f"patrol_briefing_{qdate}_{shift}.csv")
        all_results[all_results['flagged'] == 1].to_csv(export_path, index=False)
        print(f"\n✓ Flagged tiles exported to: {export_path}")

run_button.on_click(on_run)

# ── Render UI ─────────────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════╗")
print("║     VIOLENT CRIME PREDICTION — PATROL DISPATCH       ║")
print("║     XGBoost  |  Threshold 0.150  |  2.95x lift       ║")
print("╚══════════════════════════════════════════════════════╝\n")

display(
    widgets.VBox([
        widgets.HBox([date_picker, shift_dropdown]),
        top_n_slider,
        run_button,
        out
    ])
)

╔══════════════════════════════════════════════════════╗
║     VIOLENT CRIME PREDICTION — PATROL DISPATCH       ║
║     XGBoost  |  Threshold 0.150  |  2.95x lift       ║
╚══════════════════════════════════════════════════════╝



In [ ]:
# =============================================================================
# STEP 3: MAP VISUALISATION — Flagged Tiles + Beat Overlay
#
# Requires: folium (interactive map)
#   !pip install folium branca --quiet
#
# Inputs from previous cells:
#   engine          — CrimePredictionEngine instance
#   beats_gdf       — GeoDataFrame with beat boundaries (from Step 1)
#   FEATURE_COLS    — feature column list
# =============================================================================

# !pip install folium branca --quiet

import sys
import subprocess
try:
    import folium
    import folium.plugins as plugins
except ModuleNotFoundError:
    print("Installing folium and branca...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "branca", "--quiet"])
    import folium
    import folium.plugins as plugins
import h3
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape, Point
from shapely.geometry import Polygon
from IPython.display import display
import ipywidgets as widgets
from IPython.display import clear_output, HTML

# -----------------------------------------------------------------------------
# Load Chicago beat boundaries (same approach used in training notebook)
# -----------------------------------------------------------------------------
if 'beats_gdf' not in globals() or beats_gdf is None:
    print('Loading Chicago Police Beats boundary data...')
    beats_url = 'https://data.cityofchicago.org/resource/n9it-hstw.json?$limit=5000'
    beats_df = pd.read_json(beats_url)

    if 'the_geom' in beats_df.columns:
        beats_gdf = gpd.GeoDataFrame(
            beats_df.drop(columns=['the_geom']).copy(),
            geometry=beats_df['the_geom'].apply(lambda geom: shape(geom) if isinstance(geom, dict) else None),
            crs='EPSG:4326'
        )
    else:
        beats_gdf = gpd.GeoDataFrame(beats_df.copy(), geometry=None, crs='EPSG:4326')

    if 'geometry' in beats_gdf.columns:
        beats_gdf = beats_gdf[beats_gdf.geometry.notna()].copy()

    print(f'  Loaded {len(beats_gdf):,} beat boundaries.')

# -----------------------------------------------------------------------------
# Helper: H3 tile address → GeoJSON polygon
# folium expects [lat, lon] but h3 returns (lat, lon) — order is correct
# -----------------------------------------------------------------------------
def h3_to_geojson_polygon(h3_address: str) -> dict:
    boundary = h3.cell_to_boundary(h3_address)   # list of (lat, lon) tuples
    coords   = [[lon, lat] for lat, lon in boundary]
    coords.append(coords[0])                      # close the ring
    return {
        "type"       : "Feature",
        "geometry"   : { "type": "Polygon", "coordinates": [coords] },
        "properties" : { "h3_address": h3_address }
    }

# -----------------------------------------------------------------------------
# Helper: Build colour from probability score
# Low  → green (#27AE60)
# Mid  → amber (#F39C12)
# High → red   (#E74C3C)
# -----------------------------------------------------------------------------
def prob_to_hex(prob: float, threshold: float = 0.150) -> str:
    if prob < threshold:
        return "#A8D5A2"                          # soft green — below threshold
    t = min((prob - threshold) / (0.40 - threshold), 1.0)
    r = int(231 * t + 39  * (1 - t))             # interpolate red channel
    g = int(76  * t + 174 * (1 - t))             # interpolate green channel
    b = int(60  * t + 96  * (1 - t))             # interpolate blue channel
    return f"#{r:02X}{g:02X}{b:02X}"

# -----------------------------------------------------------------------------
# Core map builder
# -----------------------------------------------------------------------------
def build_patrol_map(
    results        : pd.DataFrame,       # output of engine.predict(return_all=True)
    beats_gdf,                            # GeoDataFrame from Step 1
    threshold      : float  = 0.150,
    show_all_tiles : bool   = False,      # if False, show only flagged tiles
    map_zoom       : int    = 11,
) -> folium.Map:

    # Chicago centroid
    m = folium.Map(
        location   = [41.8781, -87.6298],
        zoom_start = map_zoom,
        tiles      = "CartoDB positron",
    )

    # ── Layer 1: Beat boundaries ──────────────────────────────────────────────
    beats_layer = folium.FeatureGroup(name="Police Beats", show=True)

    beats_4326 = beats_gdf.to_crs("EPSG:4326")
    for _, beat_row in beats_4326.iterrows():
        beat_num = beat_row.get('beat_num', beat_row.get('beat', 'Unknown'))
        district = beat_row.get('district', '')
        geom     = beat_row.geometry

        folium.GeoJson(
            data       = geom.__geo_interface__,
            style_function = lambda feat: {
                "fillColor"   : "transparent",
                "color"       : "#1A237E",
                "weight"      : 1.4,
                "fillOpacity" : 0,
                "dashArray"   : "4 4",
            },
            tooltip = folium.Tooltip(
                f"<b>Beat {beat_num}</b><br>District {district}",
                sticky=False
            )
        ).add_to(beats_layer)

    beats_layer.add_to(m)

    # Map each tile to a beat using tile centroid -> polygon spatial join
    tile_points = results[['h3_address']].drop_duplicates().copy()
    tile_points[['lat', 'lon']] = tile_points['h3_address'].apply(lambda x: pd.Series(h3.cell_to_latlng(x)))
    tile_points_gdf = gpd.GeoDataFrame(
        tile_points,
        geometry=gpd.points_from_xy(tile_points['lon'], tile_points['lat']),
        crs='EPSG:4326'
    )

    beat_col = 'beat_num' if 'beat_num' in beats_4326.columns else ('beat' if 'beat' in beats_4326.columns else None)
    district_col = 'district' if 'district' in beats_4326.columns else None

    beat_lookup = {}
    if beat_col is not None:
        join_cols = [beat_col, 'geometry'] + ([district_col] if district_col else [])
        joined = gpd.sjoin(
            tile_points_gdf,
            beats_4326[join_cols],
            how='left',
            predicate='within'
        )
        for _, r in joined.iterrows():
            beat_val = r.get(beat_col)
            district_val = r.get(district_col) if district_col else None
            beat_lookup[r['h3_address']] = (
                str(beat_val) if pd.notna(beat_val) else 'Unknown',
                str(district_val) if district_val is not None and pd.notna(district_val) else ''
            )

    # ── Layer 2: Flagged tiles (dispatch recommended) ─────────────────────────
    flagged_layer  = folium.FeatureGroup(name="🚨 Flagged Tiles (Dispatch)", show=True)
    flagged_df     = results[results['flagged'] == 1].copy()

    for _, row in flagged_df.iterrows():
        prob     = float(row['crime_probability'])
        tier     = str(row['risk_tier'])
        h3_addr  = row['h3_address']
        colour   = prob_to_hex(prob, threshold)

        # Centroid for beat lookup annotation
        lat, lon = h3.cell_to_latlng(h3_addr)

        # Build tile GeoJSON polygon
        boundary = h3.cell_to_boundary(h3_addr)
        coords   = [[lon_, lat_] for lat_, lon_ in boundary]
        coords.append(coords[0])

        beat_num, district = beat_lookup.get(h3_addr, ('Unknown', ''))

        tooltip_html = (
            f"<div style='font-family:Arial;font-size:13px;'>"
            f"<b>🚨 DISPATCH RECOMMENDED</b><br>"
            f"<b>Tile:</b> {h3_addr}<br>"
            f"<b>Police beat:</b> {beat_num}" + (f" (District {district})" if district else "") + "<br>"
            f"<b>Risk tier:</b> {tier}<br>"
            f"<b>Crime probability:</b> {prob:.1%}<br>"
            f"<b>Date / Shift:</b> {row['query_date']} | "
            f"{row['shift'].replace('_',' ').title()}"
            f"</div>"
        )

        folium.Polygon(
            locations     = [(lat_, lon_) for lat_, lon_ in boundary],
            color         = colour,
            weight        = 1.5,
            fill          = True,
            fill_color    = colour,
            fill_opacity  = 0.72,
            tooltip       = folium.Tooltip(tooltip_html, sticky=True),
            popup         = folium.Popup(tooltip_html, max_width=280)
        ).add_to(flagged_layer)

    flagged_layer.add_to(m)

    # ── Layer 3: High-risk non-flagged tiles (optional monitor layer) ─────────
    monitor_layer = folium.FeatureGroup(
        name="⚠️ High-Risk Tiles (Monitor)", show=False
    )
    high_risk_df = results[
        (results['flagged'] == 0) &
        (results['crime_probability'] >= threshold * 0.65)
    ].copy()

    for _, row in high_risk_df.iterrows():
        prob    = float(row['crime_probability'])
        h3_addr = row['h3_address']
        boundary = h3.cell_to_boundary(h3_addr)

        beat_num, district = beat_lookup.get(h3_addr, ('Unknown', ''))

        tooltip_html = (
            f"<div style='font-family:Arial;font-size:12px;'>"
            f"<b>⚠️ Monitor</b><br>"
            f"Tile: {h3_addr}<br>"
            f"Police beat: {beat_num}" + (f" (District {district})" if district else "") + "<br>"
            f"Probability: {prob:.1%}<br>"
            f"(Below dispatch threshold of {threshold:.0%})"
            f"</div>"
        )

        folium.Polygon(
            locations    = [(lat_, lon_) for lat_, lon_ in boundary],
            color        = "#F39C12",
            weight       = 1.0,
            fill         = True,
            fill_color   = "#F39C12",
            fill_opacity = 0.30,
            tooltip      = folium.Tooltip(tooltip_html, sticky=True),
        ).add_to(monitor_layer)

    monitor_layer.add_to(m)

    # ── Layer control + legend ────────────────────────────────────────────────
    folium.LayerControl(position='topright', collapsed=False).add_to(m)

    # Minimap
    plugins.MiniMap(
        tile_layer  = "CartoDB positron",
        zoom_level_offset = -6,
        width=180, height=180
    ).add_to(m)

    # ── Legend HTML ───────────────────────────────────────────────────────────
    query_date = results['query_date'].iloc[0]
    shift_name = results['shift'].iloc[0].replace('_', ' ').title()
    n_flagged  = int(results['flagged'].sum())
    n_total    = len(results)
    n_critical = int((results['risk_tier'] == 'Critical').sum())
    n_high     = int((results['risk_tier'] == 'High').sum())
    n_monitor  = len(high_risk_df)

    legend_html = f"""
    <div style="
        position: fixed; bottom: 30px; left: 30px; z-index: 1000;
        background: white; border: 1px solid #ccc; border-radius: 8px;
        padding: 14px 18px; font-family: Arial; font-size: 13px;
        box-shadow: 2px 2px 8px rgba(0,0,0,0.15); min-width: 240px;
    ">
        <b style="font-size:14px; color:#1F3A6E;">
            🗺 Patrol Dispatch Map
        </b><br>
        <span style="color:#555; font-size:12px;">
            {query_date} &nbsp;|&nbsp; {shift_name}
        </span>
        <hr style="margin:8px 0; border-color:#eee;">
        <table style="width:100%; border-collapse:collapse;">
          <tr>
            <td><span style="background:#E74C3C;
                border-radius:3px; padding:2px 8px;">&nbsp;</span></td>
            <td style="padding-left:8px;">Critical (&gt;35%)</td>
            <td style="text-align:right; font-weight:bold;">{n_critical}</td>
          </tr>
          <tr>
            <td><span style="background:#E8933C;
                border-radius:3px; padding:2px 8px;">&nbsp;</span></td>
            <td style="padding-left:8px;">High (20–35%)</td>
            <td style="text-align:right; font-weight:bold;">{n_high}</td>
          </tr>
          <tr>
            <td><span style="background:#27AE60;
                border-radius:3px; padding:2px 8px;">&nbsp;</span></td>
            <td style="padding-left:8px;">Flagged total</td>
            <td style="text-align:right; font-weight:bold;">{n_flagged}</td>
          </tr>
          <tr>
            <td><span style="background:#F39C12; opacity:0.5;
                border-radius:3px; padding:2px 8px;">&nbsp;</span></td>
            <td style="padding-left:8px;">Monitor (below threshold)</td>
            <td style="text-align:right; font-weight:bold;">{n_monitor}</td>
          </tr>
          <tr>
            <td><span style="border:2px dashed #1A237E;
                border-radius:3px; padding:2px 8px;">&nbsp;</span></td>
            <td style="padding-left:8px;">Beat boundaries</td>
            <td></td>
          </tr>
        </table>
        <hr style="margin:8px 0; border-color:#eee;">
        <span style="color:#888; font-size:11px;">
            Threshold: {threshold:.0%} &nbsp;|&nbsp;
            Tiles flagged: {n_flagged}/{n_total}
            ({n_flagged/n_total:.1%})<br>
            Precision: ~19.7% &nbsp;|&nbsp; Lift: 2.95× random
        </span>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    return m


# =============================================================================
# INTERACTIVE WIDGET — date picker + shift selector → live map
# =============================================================================
date_w  = widgets.DatePicker(
    description = "Date:",
    value       = pd.Timestamp.today().date(),
    layout      = widgets.Layout(width="260px")
)
shift_w = widgets.Dropdown(
    options = [
        ("Morning / Noon  (06:00–13:59)", "morning_noon"),
        ("Afternoon / Night (14:00–21:59)", "afternoon_night"),
        ("Overnight  (22:00–05:59)", "overnight"),
    ],
    value       = "afternoon_night",
    description = "Shift:",
    layout      = widgets.Layout(width="380px")
)
show_all_w = widgets.Checkbox(
    value       = False,
    description = "Show monitor layer (below threshold)",
    layout      = widgets.Layout(width="340px")
)
run_btn = widgets.Button(
    description  = "🗺  Generate Map",
    button_style = "success",
    layout       = widgets.Layout(width="200px", height="36px")
)
map_out = widgets.Output()

def on_map_run(b):
    with map_out:
        clear_output(wait=True)

        qdate = date_w.value
        shift = shift_w.value

        if qdate is None:
            print("Please select a date.")
            return

        print(f"Scoring all tiles for {qdate} | {shift} …")

        # Score all tiles
        all_results = engine.predict(
            query_date = qdate,
            shift      = shift,
            return_all = True
        )

        n_flagged = int(all_results['flagged'].sum())
        print(f"  {n_flagged} tiles flagged for dispatch "
              f"({n_flagged/len(all_results):.1%} of all tiles)")
        print(f"  Building map …")

        patrol_map = build_patrol_map(
            results        = all_results,
            beats_gdf      = beats_gdf,
            threshold      = engine.threshold,
            show_all_tiles = show_all_w.value,
        )

        # Save and display inline
        map_path = f"patrol_map_{qdate}_{shift}.html"
        patrol_map.save(map_path)
        print(f"  ✓ Map saved → {map_path}")
        print(f"  Open the file in a browser for full interactivity.\n")

        # Display inline in notebook
        display(patrol_map)

run_btn.on_click(on_map_run)

print("╔═══════════════════════════════════════════════════════╗")
print("║    PATROL DISPATCH MAP — Flagged Tiles + Beat Overlay  ║")
print("╚═══════════════════════════════════════════════════════╝\n")
print("Layers included:")
print("  🚨 Flagged tiles   — dispatches recommended (above threshold 0.150)")
print("  ⚠️  Monitor tiles   — below threshold but elevated (toggle on)")
print("  🔵 Beat boundaries — Chicago police beat outlines\n")

display(widgets.VBox([
    widgets.HBox([date_w, shift_w]),
    show_all_w,
    run_btn,
    map_out
]))

Loading Chicago Police Beats boundary data...
  Loaded 277 beat boundaries.
╔═══════════════════════════════════════════════════════╗
║    PATROL DISPATCH MAP — Flagged Tiles + Beat Overlay  ║
╚═══════════════════════════════════════════════════════╝

Layers included:
  🚨 Flagged tiles   — dispatches recommended (above threshold 0.150)
  ⚠️  Monitor tiles   — below threshold but elevated (toggle on)
  🔵 Beat boundaries — Chicago police beat outlines

